In [1]:
import os
import json
import requests
from langchain_openai import ChatOpenAI

# 1. Config
MCP_SERVER_URL = "http://127.0.0.1:8765"
os.environ["OPENAI_API_KEY"]  # make sure your API key is set in env

# 2. Setup LLM
llm = ChatOpenAI(model="gpt-4o-mini")

In [2]:

# 3. Define helper to call MCP tools
def call_mcp_tool(tool_name: str, payload: dict):
    """Call MCP tool by name with JSON payload"""
    url = f"{MCP_SERVER_URL}/tools/{tool_name}"
    resp = requests.post(url, json=payload)
    return resp.json()


In [4]:

from langchain.agents import Tool, initialize_agent, AgentType
from typing import Dict, Any, Optional, List

_last_call = {"name": None, "arg": None}
def mcp_invoke(tool: str, inp: str) -> str:
    """
    Invoke an MCP tool with anti-repeat protection
    """
    # Prevent repeated identical tool calls
    if _last_call["name"] == tool and _last_call["arg"] == inp:
        return f"(skipped duplicate call to {tool} with same input)"
    _last_call["name"], _last_call["arg"] = tool, inp

    try:
        r = requests.post(f"{MCP_SERVER_URL}/invoke", json={"tool": tool, "input": inp})
        r.raise_for_status()
        response = r.json()
        
        if response.get("ok"):
            return response["result"]
        return f"Error: {response.get('error')}"
    except Exception as e:
        return f"Error invoking MCP tool: {str(e)}"

def load_tools_from_mcp() -> List[Tool]:
    """
    Load available tools from MCP server
    """
    try:
        resp = requests.get(f"{MCP_SERVER_URL}/tools")
        resp.raise_for_status()
        tools = []
        
        for name in resp.json().get("tools", []):
            tools.append(
                Tool(
                    name=name,
                    func=lambda arg, _name=name: mcp_invoke(_name, arg),
                    description=(
                        f"Remote MCP tool '{name}'. Call at most once per unique input. "
                        "Return value is final; do not re-call with the same arguments."
                    ),
                )
            )
        return tools
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Failed to load MCP tools: {str(e)}")


In [5]:
def api_caller(endpoint: str, payload: dict):
    """
    Generic function to call an API.
    `endpoint` = the REST endpoint (relative to localhost:8000)
    `payload` = dict body
    """
    url = f"http://localhost:8000{endpoint}"
    res = requests.post(url, json=payload)
    try:
        return res.json()
    except Exception:
        return {"status": res.status_code, "response": res.text}

In [20]:
from pydantic import BaseModel
from langchain.tools import StructuredTool


class OutcomeDef(BaseModel):
    name: str
    description: Optional[str]
    nextTask: Optional[str]

class ContextDef(BaseModel):
    key: str
    description: str
    dataType: str


class TaskDef(BaseModel):
    taskName: str
    taskDetails: str
    taskType: str
    assignmentType: str
    sequence: int
    outcomes: List[OutcomeDef]


class ConfigureWorkflowInput(BaseModel):
    workflowName: str
    description: str
    app: str
    contexts: List[ContextDef]
    tasks: List[TaskDef]
    startInstance: bool
    initialContext: Optional[Dict[str, Any]]

def configure_workflow_func(**kwargs):
    return api_caller("/admin/configure_workflow", kwargs)


# Tool 2: Complete Task
class CompleteTaskInput(BaseModel):
    taskId: int
    outcome: str

def complete_task_func(**kwargs):
    return api_caller("/admin/complete_task", kwargs)

# ================================
# 4. Register Tools
# ================================
configure_workflow_tool = StructuredTool.from_function(
    func=configure_workflow_func,
    name="configure_workflow",
    description="Configure a workflow in Orian",
    args_schema=ConfigureWorkflowInput
)

complete_task_tool = StructuredTool.from_function(
    func=complete_task_func,
    name="complete_task",
    description="Complete a task in a workflow by taskId and outcome",
    args_schema=CompleteTaskInput
)


In [ ]:
# form ConfigureWorkflowInput based on the folowign json structure BaseModel  
{
  "workflowName": "SampleWorkflow2",
  "description": "Sample workflow for demonstration",
  "app": "Agent Orc",
  "contexts": [
    {
      "key": "REQUESTER",
      "description": "Person who requested the workflow",
      "dataType": "STRING"
    }
  ],
  "tasks": [
    {
      "name": "SUBMIT_REQUEST",
      "description": "Submit initial request",
      "taskType": "MANUAL",
      "assignmentType": "USER",
      "sequence": 1,
      "outcomes": [
        {
          "name": "SUBMITTED",
          "description": "Request has been submitted",
          "nextTask": "REVIEW_REQUEST"
        }
      ]
    },
    {
      "name": "REVIEW_REQUEST",
      "description": "Review the submitted request",
      "taskType": "MANUAL",
      "assignmentType": "GROUP",
      "sequence": 2,
      "outcomes": [
        {
          "name": "APPROVED",
          "nextTask": null
        },
        {
          "name": "NEEDS_INFO",
          "nextTask": "SUBMIT_REQUEST"
        }
      ]
    }
  ],
  "startInstance": false,
  "initialContext": {
    "REQUESTER": "alice@example.com"
  }
} 
# ================================


 


In [26]:
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType

# LLM
llm = ChatOpenAI(model="gpt-4o-mini")

# Suppose you already have your tools as `mcp_tools = [tool1, tool2]`
agent = initialize_agent(
    tools=load_tools_from_mcp(),
    # tools=[configure_workflow_tool, complete_task_tool],
    llm=llm,
    agent=AgentType.OPENAI_MULTI_FUNCTIONS,  # lets LLM decide which tool
    verbose=True
)

query = "You are a workflow orchestration system. Create a workflow in Orian  named 'Document Approval' with tasks 'Submit Document', 'Review Document', and 'Approve Document'." \
" The workflow should start with 'Submit Document', followed by 'Review Document', and finally 'Approve Document'. The possible outcomes for each task are 'approved' and 'rejected'. Configure the workflow accordingly."

structured_prompt = f"""
    Based on this user request: "{query}"
    
    Generate a workflow configuration that exactly matches these Pydantic models and trigger the configuration using the `configure_workflow` tool:

    class OutcomeDef(BaseModel):
        name: str
        description: Optional[str]
        nextTask: Optional[str]

    class ContextDef(BaseModel):
        key: str
        description: str
        dataType: str

    class TaskDef(BaseModel):
        taskName: str
        taskDetails: str
        taskType: str
        assignmentType: str
        sequence: int
        outcomes: List[OutcomeDef]

    class ConfigureWorkflowInput(BaseModel):
        workflowName: str
        description: str
        app: str
        contexts: List[ContextDef]
        tasks: List[TaskDef]
        startInstance: bool
        initialContext: Optional[Dict[str, Any]]

    Rules:
    1. Response must be ONLY the valid JSON configuration
    2. taskType should be "MANUAL"
    3. assignmentType should be "USER"
    4. Include REQUESTER in contexts
    5. Connect tasks using nextTask in outcomes
    6. Task sequences should start from 1
    7. Each task must have at least one outcome
    """
result = agent.run(structured_prompt)
print(result)




> Entering new AgentExecutor chain...

Invoking: `configure_workflow` with `{'workflowName': 'Document Approval', 'description': 'A workflow to approve documents.', 'app': 'Orian', 'contexts': [{'key': 'REQUESTER', 'description': 'The person submitting the document', 'dataType': 'string'}], 'tasks': [{'taskName': 'Submit Document', 'taskDetails': 'Submit a new document for approval.', 'taskType': 'MANUAL', 'assignmentType': 'USER', 'sequence': 1, 'outcomes': [{'name': 'approved', 'description': 'Document submitted successfully.', 'nextTask': 'Review Document'}, {'name': 'rejected', 'description': 'Document submission rejected.', 'nextTask': None}]}, {'taskName': 'Review Document', 'taskDetails': 'Review the submitted document.', 'taskType': 'MANUAL', 'assignmentType': 'USER', 'sequence': 2, 'outcomes': [{'name': 'approved', 'description': 'Document approved after review.', 'nextTask': 'Approve Document'}, {'name': 'rejected', 'description': 'Document review rejected.', 'nextTask': No

TypeError: Object of type ContextDef is not JSON serializable

In [ ]:
# Create workflow from query using proper serialization
def create_workflow(query: str):
    # Create context definition
    requester_context = ContextDef(
        key="REQUESTER",
        description="Person who requested the workflow",
        dataType="STRING"
    ).model_dump()  # Convert to dict

    # Parse the tasks from query
    if "onboarding" in query.lower() and "offboarding" in query.lower():
        # Create onboarding outcomes
        onboarding_outcomes = [
            OutcomeDef(
                name="COMPLETED",
                description="Task completed successfully",
                nextTask="OFFBOARDING"
            ).model_dump(),
            OutcomeDef(
                name="REJECTED",
                description="Task was rejected",
                nextTask=None
            ).model_dump()
        ]

        # Create offboarding outcomes
        offboarding_outcomes = [
            OutcomeDef(
                name="COMPLETED",
                description="Task completed successfully",
                nextTask=None
            ).model_dump(),
            OutcomeDef(
                name="REJECTED",
                description="Task was rejected",
                nextTask=None
            ).model_dump()
        ]

        # Create tasks
        tasks = [
            TaskDef(
                taskName="ONBOARDING",
                taskDetails="Process new employee onboarding",
                taskType="MANUAL",
                assignmentType="USER",
                sequence=1,
                outcomes=onboarding_outcomes
            ).model_dump(),
            TaskDef(
                taskName="OFFBOARDING",
                taskDetails="Process employee offboarding",
                taskType="MANUAL",
                assignmentType="USER",
                sequence=2,
                outcomes=offboarding_outcomes
            ).model_dump()
        ]

        # Create the complete workflow configuration
        workflow_config = ConfigureWorkflowInput(
            workflowName="Employee Processing Flow",
            description="Handle employee onboarding and offboarding process",
            app="Agent Orc",
            contexts=[requester_context],
            tasks=tasks,
            startInstance=False,
            initialContext={"REQUESTER": "user@example.com"}
        )

        # Call the configuration function with the model data
        result = configure_workflow_func(**workflow_config.model_dump())
        return result

# Test the workflow creation
query = "create a workflow with 2 tasks, onboarding and offboarding. Once onboarding completed offboard should be the next task"
result = create_workflow(query)
print("Workflow Configuration Result:")
print(json.dumps(result, indent=2))

In [24]:
# Function to generate workflow configuration using LLM
def generate_workflow_from_query(query: str):
    # Create a prompt that explains the required structure
    structured_prompt = f"""
    Based on this user request: "{query}"
    
    Generate a workflow configuration that exactly matches these Pydantic models:

    class OutcomeDef(BaseModel):
        name: str
        description: Optional[str]
        nextTask: Optional[str]

    class ContextDef(BaseModel):
        key: str
        description: str
        dataType: str

    class TaskDef(BaseModel):
        taskName: str
        taskDetails: str
        taskType: str
        assignmentType: str
        sequence: int
        outcomes: List[OutcomeDef]

    class ConfigureWorkflowInput(BaseModel):
        workflowName: str
        description: str
        app: str
        contexts: List[ContextDef]
        tasks: List[TaskDef]
        startInstance: bool
        initialContext: Optional[Dict[str, Any]]

    Rules:
    1. Response must be ONLY the valid JSON configuration
    2. taskType should be "MANUAL"
    3. assignmentType should be "USER"
    4. Include REQUESTER in contexts
    5. Connect tasks using nextTask in outcomes
    6. Task sequences should start from 1
    7. Each task must have at least one outcome
    """

    # Get the configuration from LLM
    result = llm.invoke(structured_prompt)
    
    try:
        # Parse the LLM response to get just the JSON
        config_dict = json.loads(result.content)
        
        # Validate using Pydantic model
        config = ConfigureWorkflowInput(**config_dict)
        
        # Call the workflow configuration function
        result = configure_workflow_func(**config.model_dump())
        return {"status": "success", "result": result, "config": config.model_dump()}
    except Exception as e:
        return {"status": "error", "error": str(e)}

# Test with a sample query
query = "create a workflow with 2 tasks, onboarding and offboarding. Once onboarding completed offboard should be the next task"
result = generate_workflow_from_query(query)
print("Generated Workflow Configuration:")
print(json.dumps(result, indent=2))

Generated Workflow Configuration:
{
  "status": "error",
  "error": "Expecting value: line 1 column 1 (char 0)"
}


In [ ]:
# Example of expected workflow configuration
example_config = {
    "workflowName": "Employee Processing",
    "description": "Handle employee onboarding and offboarding process",
    "app": "Agent Orc",
    "contexts": [
        {
            "key": "REQUESTER",
            "description": "Person who initiated the process",
            "dataType": "STRING"
        }
    ],
    "tasks": [
        {
            "taskName": "ONBOARDING",
            "taskDetails": "Complete employee onboarding process",
            "taskType": "MANUAL",
            "assignmentType": "USER",
            "sequence": 1,
            "outcomes": [
                {
                    "name": "COMPLETED",
                    "description": "Onboarding process completed successfully",
                    "nextTask": "OFFBOARDING"
                },
                {
                    "name": "REJECTED",
                    "description": "Onboarding process was rejected",
                    "nextTask": None
                }
            ]
        },
        {
            "taskName": "OFFBOARDING",
            "taskDetails": "Process employee offboarding",
            "taskType": "MANUAL",
            "assignmentType": "USER",
            "sequence": 2,
            "outcomes": [
                {
                    "name": "COMPLETED",
                    "description": "Offboarding process completed",
                    "nextTask": None
                },
                {
                    "name": "REJECTED",
                    "description": "Offboarding process was rejected",
                    "nextTask": None
                }
            ]
        }
    ],
    "startInstance": False,
    "initialContext": {
        "REQUESTER": "user@example.com"
    }
}

# Test the configuration
result = agent.run("Configure a workflow using this exact structure: " + json.dumps(example_config, indent=2))
print(result)